# 02 · ReAct Practice Cookbook (Lots of Examples)

## 🎯 What You'll Learn
- Advanced tool selection strategies and guardrails
- Error handling and edge case management
- 10+ realistic legal scenarios to practice
- Prompt engineering patterns for better reasoning
- Agent validation and quality assurance

## 📋 Shared Legal Dataset
Using the same contracts as Notebook 1 for consistency:

```python
CONTRACT_A = "EMPLOYMENT AGREEMENT between TechMahindra Solutions and John Doe. Termination: Either party may terminate with 30 days notice. Liability: Company liable for damages up to $50,000."
CONTRACT_B = "SERVICE AGREEMENT between ABC Corp and XYZ Ltd. Payment terms: Net 30 days. Governing law: This agreement shall be governed by Maharashtra state laws."
```

Master advanced ReAct patterns through intensive hands-on practice!


## Theory: Prompt patterns that help ReAct
- State goal → list options → pick one → say why → act.
- Keep reasoning short but clear; avoid hedging.


In [7]:
# Shared dataset for all exercises
CONTRACT_A = "EMPLOYMENT AGREEMENT between TechMahindra Solutions and John Doe. Termination: Either party may terminate with 30 days notice. Liability: Company liable for damages up to $50,000."
CONTRACT_B = "SERVICE AGREEMENT between ABC Corp and XYZ Ltd. Payment terms: Net 30 days. Governing law: This agreement shall be governed by Maharashtra state laws."

# 10 Realistic Legal Scenarios for Practice
PRACTICE_SCENARIOS = [
    # Basic scenarios
    ("Hello, I need help with my contract", "greet"),
    ("What does the termination clause mean?", "explain"),
    ("Find liability information", "search"),
    
    # Edge cases
    ("", "greet"),  # Empty query
    ("asdfghjkldfdsdfsdfsdfsdffffffffffffff", "search"),  # Gibberish
    ("HELLO THERE!!!!!!!!!!!!!!!!", "greet"),  # Caps + punctuation
    
    # Complex legal queries
    ("Is there any termination penalty I should worry about?", "search"),
    ("Can you explain the payment terms in simple language?", "explain"),
    ("What are the liability risks for me in this employment agreement?", "search"),
    ("Help me understand what governing law means", "explain"),
    
    # Ambiguous cases (multiple valid answers)
    ("Tell me about termination", "search"),  # Could be search or explain
    ("Payment information please", "search"),
]

# Enhanced exercise runner with validation
def run_exercises(agent_fn, show_details=True):
    """Run agent through all practice scenarios with validation"""
    results = []
    
    print("🧪 RUNNING 12 PRACTICE SCENARIOS")
    print("=" * 50)
    
    for i, (query, expected_action) in enumerate(PRACTICE_SCENARIOS, 1):
        try:
            result = agent_fn(query)
            actual_action = result.get("action", "unknown")
            
            # Validation
            is_correct = actual_action == expected_action
            status = "✅ PASS" if is_correct else "❌ FAIL"
            
            if show_details:
                print(f"\n{i:2d}. {status} '{query}'")
                print(f"    Expected: {expected_action} | Got: {actual_action}")
                if not is_correct:
                    print(f"    Reasoning: {result.get('reasoning', 'N/A')}")
            
            results.append({
                "scenario": i,
                "query": query,
                "expected": expected_action,
                "actual": actual_action,
                "correct": is_correct,
                "result": result
            })
            
        except Exception as e:
            print(f"\n{i:2d}. 💥 ERROR '{query}' - {str(e)}")
            results.append({
                "scenario": i,
                "query": query,
                "expected": expected_action,
                "actual": "ERROR",
                "correct": False,
                "error": str(e)
            })
    
    # Summary
    correct_count = sum(1 for r in results if r.get("correct", False))
    total_count = len(results)
    score = (correct_count / total_count) * 100
    
    print(f"\n📊 FINAL SCORE: {correct_count}/{total_count} ({score:.1f}%)")
    
    if score >= 90:
        print("🏆 EXCELLENT! Your agent handles edge cases well!")
    elif score >= 75:
        print("👍 GOOD! A few edge cases need attention.")
    else:
        print("🔧 NEEDS WORK. Focus on rule refinement and error handling.")
    
    return results


## Advanced Tool Selection with Guardrails

Let's build a robust agent that handles edge cases gracefully:


In [10]:
from typing import Dict, Optional

def safe_tool_search(query: str) -> str:
    """Enhanced search tool with keyword matching"""
    if not query or not query.strip():
        return "[search] Please provide a specific question about the contract."
    
    q = query.lower()
    
    # Legal term matching
    if any(term in q for term in ["termination", "terminate", "end", "quit"]):
        return "[search] Found: Either party may terminate with 30 days notice"
    if any(term in q for term in ["payment", "pay", "money", "fee"]):
        return "[search] Found: Payment terms: Net 30 days"
    if any(term in q for term in ["liability", "liable", "damage", "risk"]):
        return "[search] Found: Company liable for damages up to $50,000"
    if any(term in q for term in ["law", "governing", "jurisdiction"]):
        return "[search] Found: Governed by Maharashtra state laws"
    
    # Fallback for unmatched queries
    return f"[search] Searched contract for: {query[:50]}..."

def safe_tool_explain(query: str) -> str:
    """Enhanced explain tool with better responses"""
    if not query or not query.strip():
        return "[explain] I'd be happy to explain any legal terms. What would you like to know?"
    
    return f"[explain] In simple words: {query[:60]}... (This is a legal term that means...)"

def safe_tool_greet(name: str = "Friend") -> str:
    """Enhanced greet tool"""
    return f"[greet] Hello {name}! I'm your legal assistant. Upload a contract and I'll help analyze it for risks, terms, and obligations."

# Advanced tool selection with guardrails
def advanced_choose_tool(user_query: str):
    """Enhanced tool selection with error handling and guardrails"""
    
    # Handle empty/None queries
    if not user_query or not user_query.strip():
        return "greet", lambda: safe_tool_greet()
    
    # Normalize query
    q = user_query.lower().strip()
    
    # Remove excessive punctuation and normalize
    import re
    q_clean = re.sub(r'[!]{2,}', '!', q)  # Multiple exclamation marks
    q_clean = re.sub(r'[?]{2,}', '?', q_clean)  # Multiple question marks
    
    # Greeting patterns (high priority)
    greeting_patterns = ["hi", "hello", "hey", "greetings", "good morning", "good afternoon"]
    if any(pattern in q_clean for pattern in greeting_patterns):
        return "greet", lambda: safe_tool_greet()
    
    # Explanation patterns
    explain_patterns = ["explain", "meaning", "mean", "what is", "what does", "simplify", "understand"]
    if any(pattern in q_clean for pattern in explain_patterns):
        return "explain", lambda: safe_tool_explain(user_query)
    
    # Search patterns (default but enhanced)
    return "search", lambda: safe_tool_search(user_query)

# Enhanced reasoning templates with context awareness
ADVANCED_REASONING_TEMPLATES = {
    "greet": "The user is greeting me or needs initial guidance, so I should respond warmly and offer help.",
    "search": "The user wants specific information from the contract, so I need to search for relevant terms and clauses.",
    "explain": "The user needs clarification or simplification of legal language, so I should provide a clear explanation."
}

def advanced_react_agent(user_query: str) -> Dict[str, str]:
    """Enhanced single-step ReAct agent with guardrails and error handling"""
    
    try:
        # REASONING: Enhanced decision making
        action, tool_fn = advanced_choose_tool(user_query)
        
        # Context-aware reasoning
        reasoning = ADVANCED_REASONING_TEMPLATES.get(action, f"I will use {action} to help with: {user_query}")
        
        # Add query analysis to reasoning for complex cases
        if user_query and len(user_query.strip()) > 50:
            reasoning += " This is a complex query that needs careful analysis."
        elif not user_query or not user_query.strip():
            reasoning = "The user provided an empty query, so I'll offer a friendly greeting and guidance."
        
        # ACTION: Execute tool with error handling
        try:
            observation = tool_fn()
        except Exception as tool_error:
            observation = f"[error] Tool execution failed: {str(tool_error)}"
            action = "error"
        
        # RESPONSE: Enhanced final answer
        if action == "error":
            answer = "I encountered an error processing your request. Please try rephrasing your question."
        else:
            answer = observation
        
        return {
            "reasoning": reasoning,
            "action": action,
            "observation": observation,
            "answer": answer,
            "query_length": len(user_query) if user_query else 0,
            "has_legal_terms": any(term in user_query.lower() for term in ["contract", "legal", "clause", "term"] if user_query)
        }
        
    except Exception as e:
        # Top-level error handling
        return {
            "reasoning": f"Critical error in agent processing: {str(e)}",
            "action": "error",
            "observation": f"[error] {str(e)}",
            "answer": "I'm sorry, I encountered an unexpected error. Please try again.",
            "query_length": 0,
            "has_legal_terms": False
        }

# Test the enhanced agent
print("🤖 TESTING ADVANCED REACT AGENT")
print("=" * 40)

# Quick test with different types of queries
test_queries = [
    "Hello there!",
    "What does termination mean?", 
    "Find liability risks",
    "",  # Empty query
    "HELP ME!!!",  
    "abcdefghijklmnopqrstuvwxyz",
    "HI!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!",
    "                       "
]

for query in test_queries:
    result = advanced_react_agent(query)
    print(f"\n📝 Query: '{query}'")
    print(f"🧠 Reasoning: {result['reasoning']}")
    print(f"⚡ Action: {result['action']}")
    print(f"💬 Answer: {result['answer']}")
    print("-" * 30)


🤖 TESTING ADVANCED REACT AGENT

📝 Query: 'Hello there!'
🧠 Reasoning: The user is greeting me or needs initial guidance, so I should respond warmly and offer help.
⚡ Action: greet
💬 Answer: [greet] Hello Friend! I'm your legal assistant. Upload a contract and I'll help analyze it for risks, terms, and obligations.
------------------------------

📝 Query: 'What does termination mean?'
🧠 Reasoning: The user needs clarification or simplification of legal language, so I should provide a clear explanation.
⚡ Action: explain
💬 Answer: [explain] In simple words: What does termination mean?... (This is a legal term that means...)
------------------------------

📝 Query: 'Find liability risks'
🧠 Reasoning: The user wants specific information from the contract, so I need to search for relevant terms and clauses.
⚡ Action: search
💬 Answer: [search] Found: Company liable for damages up to $50,000
------------------------------

📝 Query: ''
🧠 Reasoning: The user provided an empty query, so I'll offe

## 🧪 Practice Exercise 1: Run Full Test Suite

Test your advanced agent against all 12 scenarios:


In [11]:
# Run the comprehensive test suite
results = run_exercises(advanced_react_agent)

# Analyze failures (if any)
failures = [r for r in results if not r.get("correct", False)]
if failures:
    print(f"\n🔍 ANALYZING {len(failures)} FAILURES:")
    for f in failures:
        print(f"  • '{f['query']}' → Expected: {f['expected']}, Got: {f['actual']}")
else:
    print("\n🎉 PERFECT SCORE! All scenarios handled correctly!")


🧪 RUNNING 12 PRACTICE SCENARIOS

 1. ✅ PASS 'Hello, I need help with my contract'
    Expected: greet | Got: greet

 2. ✅ PASS 'What does the termination clause mean?'
    Expected: explain | Got: explain

 3. ✅ PASS 'Find liability information'
    Expected: search | Got: search

 4. ✅ PASS ''
    Expected: greet | Got: greet

 5. ✅ PASS 'asdfghjkldfdsdfsdfsdfsdffffffffffffff'
    Expected: search | Got: search

 6. ✅ PASS 'HELLO THERE!!!!!!!!!!!!!!!!'
    Expected: greet | Got: greet

 7. ✅ PASS 'Is there any termination penalty I should worry about?'
    Expected: search | Got: search

 8. ✅ PASS 'Can you explain the payment terms in simple language?'
    Expected: explain | Got: explain

 9. ❌ FAIL 'What are the liability risks for me in this employment agreement?'
    Expected: search | Got: greet
    Reasoning: The user is greeting me or needs initial guidance, so I should respond warmly and offer help. This is a complex query that needs careful analysis.

10. ✅ PASS 'Help me und

## 🧪 Practice Exercise 2: Build Your Own Agent

Your turn! Create an agent that beats the test suite:


In [12]:
def my_custom_agent(user_query: str) -> Dict[str, str]:
    """
    YOUR CHALLENGE: Build an agent that scores 100% on the test suite!
    
    Requirements:
    - Handle empty queries gracefully
    - Recognize greetings, explanations, and search requests
    - Include proper error handling
    - Return dict with: reasoning, action, observation, answer
    """
    
    # TODO: Implement your logic here
    # Hints:
    # 1. Start with input validation
    # 2. Use pattern matching for tool selection  
    # 3. Add try/except for error handling
    # 4. Test against edge cases
    
    return {
        "reasoning": "I need to implement this logic",
        "action": "greet",  # Change this!
        "observation": "Not implemented yet",
        "answer": "Please implement the agent logic"
    }

# Test your agent (should score 100%!)
print("🎯 TESTING YOUR CUSTOM AGENT:")
my_results = run_exercises(my_custom_agent, show_details=False)

# Challenge: Can you beat the advanced_react_agent?


🎯 TESTING YOUR CUSTOM AGENT:
🧪 RUNNING 12 PRACTICE SCENARIOS

📊 FINAL SCORE: 3/12 (25.0%)
🔧 NEEDS WORK. Focus on rule refinement and error handling.


## 🧪 Practice Exercise 3: Prompt Engineering Challenge

Experiment with different reasoning patterns:


In [ ]:
# Three different reasoning styles to experiment with

REASONING_STYLE_A = {
    "greet": "User greeting detected. Responding with warm welcome.",
    "search": "Information request identified. Searching contract database.",
    "explain": "Clarification needed. Providing simple explanation."
}

REASONING_STYLE_B = {
    "greet": "The user is saying hello, so I should be friendly and helpful.",
    "search": "The user wants to find something in the contract, so I'll search for it.",
    "explain": "The user doesn't understand something, so I'll explain it clearly."
}

REASONING_STYLE_C = {
    "greet": "Greeting pattern matched → activate welcome protocol",
    "search": "Query contains search intent → execute document retrieval",
    "explain": "Explanation request detected → initiate simplification mode"
}

def create_agent_with_style(reasoning_style):
    """Factory function to create agents with different reasoning styles"""
    
    def styled_agent(user_query: str) -> Dict[str, str]:
        try:
            action, tool_fn = advanced_choose_tool(user_query)
            reasoning = reasoning_style.get(action, f"Using {action} for: {user_query}")
            observation = tool_fn()
            
            return {
                "reasoning": reasoning,
                "action": action,
                "observation": observation,
                "answer": observation
            }
        except Exception as e:
            return {
                "reasoning": f"Error: {str(e)}",
                "action": "error",
                "observation": f"[error] {str(e)}",
                "answer": "Sorry, an error occurred."
            }
    
    return styled_agent

# Test all three styles
styles = {
    "Style A (Formal)": REASONING_STYLE_A,
    "Style B (Conversational)": REASONING_STYLE_B, 
    "Style C (Technical)": REASONING_STYLE_C
}

print("🎨 COMPARING REASONING STYLES")
print("=" * 50)

test_query = "What does the liability clause mean?"

for style_name, style_dict in styles.items():
    agent = create_agent_with_style(style_dict)
    result = agent(test_query)
    
    print(f"\n{style_name}:")
    print(f"  Reasoning: {result['reasoning']}")
    print(f"  Action: {result['action']}")
    print(f"  Answer: {result['answer'][:60]}...")

print("\n🤔 REFLECTION:")
print("Which reasoning style feels most natural?")
print("Which would be clearer to end users?")
print("Which would be easier for debugging?")


## 📊 Final Assessment & Key Takeaways

**Acceptance Criteria for Notebook 2:**
- ✅ Your custom agent scores 90%+ on the test suite
- ✅ You understand how guardrails prevent agent failures
- ✅ You can explain the trade-offs between different reasoning styles
- ✅ You've handled edge cases like empty queries and errors

**What You've Mastered:**
1. **Robust Tool Selection**: Advanced pattern matching with fallbacks
2. **Error Handling**: Graceful degradation when things go wrong
3. **Edge Case Management**: Empty queries, gibberish, formatting issues
4. **Prompt Engineering**: Different reasoning styles for different contexts
5. **Systematic Testing**: Comprehensive validation of agent behavior

**Connection to Full Project:**
- These guardrails become essential when using LLMs (which can be unpredictable)
- The test suite approach scales to production agent validation
- Error handling patterns apply to multi-step reasoning chains
- Reasoning style experiments inform LLM prompt design

**Next Up:** Notebook 3 - Multi-step reasoning with LangGraph workflows!
